In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

In [7]:
def load_and_preprocess_data(filepath='dataset.csv'):
    """Load and preprocess the dataset"""
    print("📊 Loading dataset...")
    df = pd.read_csv(filepath)
    
    # Drop ID column
    df = df.drop('id', axis=1)
    
    # Handle missing values
    df['bmi'] = df['bmi'].fillna(df['bmi'].median())
    df['smoking_status'] = df['smoking_status'].fillna(df['smoking_status'].mode()[0])
    
    # Encode categorical columns
    categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
    label_encoders = {}
    
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le
    
    return df, label_encoders

In [8]:
# Define model with hyperparameters
RANDOM_STATE = 42
TEST_SIZE = 0.2
def train_model(X_train, y_train, X_test, y_test):
    """Train and evaluate models"""
    print("\n🤖 Training Random Forest Classifier...")
    
    # Define model with hyperparameters
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    # Train the model
    rf.fit(X_train, y_train)
    
    # Evaluate
    y_pred = rf.predict(X_test)
    y_pred_proba = rf.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"✅ Model trained successfully!")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   ROC-AUC: {roc_auc:.4f}")
    
    return rf

In [9]:
def save_model(model, scaler, feature_columns, label_encoders, filepath='stroke_model.pkl'):
    """Save model and preprocessing objects"""
    model_data = {
        'model': model,
        'scaler': scaler,
        'feature_columns': feature_columns,
        'label_encoders': label_encoders,
        'model_name': 'Random Forest'
    }
    
    with open(filepath, 'wb') as f:
        pickle.dump(model_data, f)
    
    print(f"\n💾 Model saved to {filepath}")
    return filepath

In [12]:
def main():
    """Main execution function"""
    print("=" * 50)
    print("🧠 STROKE PREDICTION MODEL TRAINING")
    print("=" * 50)
    
    # Load and preprocess data
    df, label_encoders = load_and_preprocess_data()
    
    # Split features and target
    X = df.drop('stroke', axis=1)
    y = df['stroke']
    feature_columns = X.columns.tolist()
    
    print(f"\n📈 Dataset shape: {df.shape}")
    print(f"   Stroke cases: {y.sum()} ({y.sum()/len(y)*100:.2f}%)")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    
    # Apply SMOTE
    print("\n🔄 Applying SMOTE to balance classes...")
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
    # Scale features
    print("📏 Scaling features...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_resampled)
    X_test_scaled = scaler.transform(X_test)
    
    # Train model
    model = train_model(X_train_scaled, y_train_resampled, X_test_scaled, y_test)
    
    # Save model
    save_model(model, scaler, feature_columns, label_encoders)
    
    print("\n" + "=" * 50)
    print("✅ TRAINING COMPLETE!")
    print("=" * 50)
if __name__ == "__main__":
    main()

🧠 STROKE PREDICTION MODEL TRAINING
📊 Loading dataset...

📈 Dataset shape: (43400, 11)
   Stroke cases: 783 (1.80%)

🔄 Applying SMOTE to balance classes...
📏 Scaling features...

🤖 Training Random Forest Classifier...
✅ Model trained successfully!
   Accuracy: 0.8297
   ROC-AUC: 0.7619

💾 Model saved to stroke_model.pkl

✅ TRAINING COMPLETE!
